# Numerical parameter identifiability

See the [course by Marisa Eisenberg](https://epimath.org/epid-814-materials/) for an excellent introduction into the topic


<img src="assets/identifiability.png" style="max-height: 300px;">

In [ ]:
import numpy as np

from mxlpy import Model, Simulator, fns, plot
from mxlpy.identify import profile_likelihood

We start with an SIR model, which we use to generate some data (this would usually be experimentally measured data)

In [ ]:
def sir() -> Model:
    return (
        Model()
        .add_variables({"s": 0.9, "i": 0.1, "r": 0.0})
        .add_parameters({"beta": 0.2, "gamma": 0.1})
        .add_reaction(
            "infection",
            fns.mass_action_2s,
            args=["s", "i", "beta"],
            stoichiometry={"s": -1, "i": 1},
        )
        .add_reaction(
            "recovery",
            fns.mass_action_1s,
            args=["i", "gamma"],
            stoichiometry={"i": -1, "r": 1},
        )
    )


data = Simulator(sir()).simulate(100).get_result().unwrap_or_err().variables
_ = plot.lines(data)

We then, for `n` different values of each parameter we are interested in, we 

- draw random samples for the remaining model parameters 
- fit the model to the data (excluding the parameter we are interested in) and note the final error
- visualise the error for each parameter value

The error for a parameter should show a clear minimum around the different values used, otherwise it is not identifiable

In [ ]:
errors_beta = profile_likelihood(
    sir(),
    data=data,
    parameter_name="beta",
    parameter_values=np.linspace(0.2 * 0.5, 0.2 * 1.5, 10),
    n_random=10,
)

fig, ax = plot.lines(errors_beta, legend=False)
ax.set(title="beta", xlabel="parameter value", ylabel="abs(error)")
plot.show()

## Fisher information

Profile likelihoods need many fits per parameter.
A much cheaper first look is the Fisher information at the optimum, which needs a single [sensitivity](forward-sensitivities.ipynb) evaluation.
From the noise-weighted sensitivities of the residuals `S` it computes `F = SᵀS`, whose inverse is the asymptotic covariance of the parameters.
It assumes Gaussian noise and a model that is approximately linear around the optimum.

Pass the noise level as `sigma` (a scalar, one value per observable or per data point).
Without it, the noise variance is estimated from the residuals.

In [ ]:
from mxlpy import identify

fim = identify.fisher_information(
    sir(),
    data=data,
    parameters={"beta": 0.2, "gamma": 0.1},  # usually a fit result
    sigma=0.01,
)
fim.wald_ci(alpha=0.95)

The correlation matrix shows which parameters compensate for each other.

In [ ]:
fig, ax, _ = plot.heatmap(fim.correlation, annotate=True)
plot.show()

Small eigenvalues of `F` belong to *sloppy* parameter combinations, which the data barely constrains.
If we only observe `i` during the first two time units, it grows approximately with rate `0.9 * beta - gamma`.
Only this combination is determined: the eigenvalues differ by orders of magnitude, the eigenvector of the small one changes `beta` and `gamma` together in a ratio of about `1 : 0.9`, and the confidence intervals become huge.

In [ ]:
early = data.loc[:2, ["i"]]
fim = identify.fisher_information(
    sir(),
    data=early,
    parameters={"beta": 0.2, "gamma": 0.1},
    sigma=0.01,
)
eigenvalues, eigenvectors = fim.eigen()
eigenvalues

In [ ]:
eigenvectors

In [ ]:
fim.wald_ci()

If `F` is exactly singular (e.g. only the product of two parameters enters the model), `fim.non_identifiable()` returns the parameter combinations spanning its null space, and the standard errors of the parameters involved are infinite.
With `transform="log"` uncertainties are expressed in log10 units, i.e. relative to the parameter values.